# 3.0.3 - Phân chia tập Validation thành 5 Folds theo cơ chế Walk-Forward (Expanding Train)

### 🎯 Mục tiêu:
Chia tập **Validation** (`Split == 2`, gồm **1,360 quan sát**, giai đoạn `2015-06-10` → `2020-10-30`) thành **5 Folds** theo cơ chế **Walk-Forward Validation** phục vụ tinh chỉnh siêu tham số (Hyperparameter Tuning) và lựa chọn mô hình:
- **Số lượng quan sát mỗi Fold:** $1,360 / 5 = 272$ ngày giao dịch cố định.
- **Cơ chế Training:** Cửa sổ mở rộng (**Expanding Window**), bắt đầu từ phiên đầu tiên của tập Train (`1990-03-27`) đến ngay trước Fold Validation tương ứng.
- **Cơ chế Validation:** Cửa sổ trượt cố định (**Fixed ~272 days**) tịnh tiến theo thời gian.

### 🛡️ Nguyên tắc chống Data Leakage:
- Không dùng K-Fold ngẫu nhiên (No Random K-Fold).
- Tập Training của mỗi fold luôn nằm hoàn toàn trước tập Validation tương ứng.
- Tập Test (`Split == 3`) hoàn toàn độc lập và không tham gia vào quá trình này.
- Cột `Val_Fold` mang giá trị `1..5` cho các dòng Validation, và `0` cho tập Train / Test.

---
- **Dữ liệu đầu vào:** `dataset/3.0.2_split.csv`
- **Dữ liệu đầu ra:** `dataset/3.0.3_val_folds.csv`

In [ ]:
import os
import numpy as np
import pandas as pd

# 1. Đọc dữ liệu từ phiên bản 3.0.2_split.csv
dataset_dir = 'dataset' if os.path.exists('dataset') else ('../dataset' if os.path.exists('../dataset') else '.')
input_path = os.path.join(dataset_dir, '3.0.2_split.csv')
print(f"Đọc dữ liệu từ: {input_path}")
df = pd.read_csv(input_path)

# Đảm bảo sắp xếp đúng thứ tự thời gian
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
print(f"Kích thước dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột.")
display(df.head(3))

In [ ]:
# 2. Xác định các chỉ số của tập Validation (Split == 2)
val_indices = df[df['Split'] == 2].index
n_val = len(val_indices)
n_folds = 5
fold_size = n_val // n_folds

print(f"Tổng số quan sát tập Validation: {n_val} dòng")
print(f"Số lượng Folds: {n_folds}")
print(f"Kích thước mỗi Fold: {fold_size} dòng/fold\n")

# Khởi tạo cột Val_Fold (0 = Không thuộc Validation, 1..5 = Fold tương ứng)
df['Val_Fold'] = 0

fold_summary = []
for f in range(1, n_folds + 1):
    f_start = (f - 1) * fold_size
    f_end = f * fold_size
    current_fold_idx = val_indices[f_start:f_end]
    df.loc[current_fold_idx, 'Val_Fold'] = f
    
    # Tập Training tương ứng (Expanding Window từ đầu đến trước fold)
    train_slice = df.loc[0 : current_fold_idx[0] - 1]
    val_slice = df.loc[current_fold_idx]
    
    fold_summary.append({
        'Fold': f,
        'Train Expanding (Start)': train_slice['Date'].iloc[0].strftime('%Y-%m-%d'),
        'Train Expanding (End)': train_slice['Date'].iloc[-1].strftime('%Y-%m-%d'),
        'Train Obs': len(train_slice),
        'Validation (Start)': val_slice['Date'].iloc[0].strftime('%Y-%m-%d'),
        'Validation (End)': val_slice['Date'].iloc[-1].strftime('%Y-%m-%d'),
        'Val Obs': len(val_slice)
    })

summary_df = pd.DataFrame(fold_summary)
print("=== BẢNG THÔNG SỐ CHI TIẾT 5 FOLDS WALK-FORWARD VALIDATION ===")
display(summary_df)

In [ ]:
# 3. Thống kê số lượng dòng theo Val_Fold và phân phối Volatility_Regime
print("Phân phối số quan sát theo Val_Fold:")
print(df['Val_Fold'].value_counts().sort_index())

print("\nPhân phối Volatility_Regime qua từng Fold Validation:")
val_regime = pd.crosstab(
    df[df['Split'] == 2]['Val_Fold'].map(lambda x: f'Fold {x}'),
    df[df['Split'] == 2]['Volatility_Regime'],
    margins=True
)
display(val_regime)

In [ ]:
# 4. Hàm mẫu minh họa cách duyệt qua 5 Fold trong quá trình huấn luyện mô hình
def get_walk_forward_cv(df_data):
    """
    Generator trả về (train_idx, val_idx) cho từng Fold
    - train_idx: Expanding window từ dòng 0 đến trước fold hiện tại
    - val_idx: Dòng thuộc fold hiện tại
    """
    for fold_num in range(1, 6):
        val_mask = (df_data['Val_Fold'] == fold_num)
        val_idx = df_data[val_mask].index.values
        train_idx = np.arange(0, val_idx[0])
        yield fold_num, train_idx, val_idx

print("Kiểm tra generator duyệt 5 folds:")
for fold_num, train_idx, val_idx in get_walk_forward_cv(df):
    print(f"Fold {fold_num}: Train idx [0:{train_idx[-1]}] ({len(train_idx)} dòng) | Val idx [{val_idx[0]}:{val_idx[-1]}] ({len(val_idx)} dòng)")

In [ ]:
# 5. Xuất file kết quả dataset/3.0.3_val_folds.csv
output_path = os.path.join(dataset_dir, '3.0.3_val_folds.csv')
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
df.to_csv(output_path, index=False)

print(f"✅ ĐÃ XUẤT THÀNH CÔNG: {output_path}")
print(f"Kích thước Dataset mới: {df.shape[0]} dòng, {df.shape[1]} cột (đã thêm cột Val_Fold).")
display(df[['Date', 'Close', 'Return_1D', 'Volatility_Regime', 'Split', 'Val_Fold']].iloc[6348:6356])
